# COLAB compatible clustering tool.



In [1]:
repo = 'https://github.com/DariahBE/clustering_assisted_gui.git'
!git clone {repo}

Cloning into 'clustering_assisted_gui'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 46 (delta 11), reused 40 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 127.78 KiB | 14.20 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [1]:
import os
import sys
import pandas as pd

sys.path.append('clustering_assisted_gui/utils')

In [2]:
import gpu_manager  #hardware interaction
import callables as c   # getters/setters for pipeline
## multiple 'steps' each have their own layer ==> data excahnge using callables
from vectorization_layer import TransformerGUI
from dimred_layer import DimensionalityReductionGUI
from clustering_layer import Clustermachine
from noise_extension_layer import NoiseExtender
from dimviz import DimensionVisualizer
from cluster_inspector import ClusterInspectorGUI



In [ ]:
max_gpus = 1        #(INT): how many GPUS is this notebook allowed to use at most? 

######### Leave this block of code as is: #########
gpu_manager.pick_gpu(1, 'auto', int(max_gpus))
os.environ["TOKENIZERS_PARALLELISM"] = "true"
devices = gpu_manager.pick_gpu(mode = 'report', verbosity=0)
if not bool(devices): 
    device = 'cpu'
else:
    device = 'cuda'

print(f"Using {device}")

In [6]:
TransformerGUI(
    config_path = "./clustering_assisted_gui/transformerconfig.json",
    device = device,
    on_result = c.receive_embeddings
).display()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

<function receive_embeddings at 0x79265e36f420>


In [ ]:
#BUG: visualization is not showing in google Colab. ==> applies to all subsequent visualizations.
DimensionalityReductionGUI(
    c.fetch_vectors,
    config_path = "clustering_assisted_gui/dimredconfig.json",
    on_result = c.receive_reduced
).display()

In [8]:
DimensionVisualizer(
    c.get_reduced,
    normalize=True
).display()

In [10]:
Clustermachine(
    c.get_reduced,
    config_path = "clustering_assisted_gui/clusterconfig.json",
    on_result = c.receive_vectorlabels,
    stringgetter = c.get_input_list
).display()

In [11]:
NoiseExtender(
    c.get_reduced,
    c.get_cluster_labels,
    "clustering_assisted_gui/noise_extension_methods.json",
    c.receive_denoised_results
).display()

In [12]:
ClusterInspectorGUI(
    c.get_input_list,
    c.get_reduced,
    c.get_cluster_labels,
    c.get_denoised_labels,
    c.get_denoised_sources
).display()